# Reconstructed jet-selection influence

Compare inclusive-jet distributions after the standard jet-ID selection (`jetId`), the track-maximum requirement only (`trkMax`), and before either selection (`noSel`). The notebook first displays the stored two-dimensional Gen, Reco, matched-Ref, and RefSel inputs, then projects $\eta^{jet}$ in each configured jet-$p_T$ interval in both the Lab and CM frames. Reco, matched Ref, RefSel, and Gen are overlaid on individual full-distribution canvases, while all ratios to Gen are overlaid on separate canvases. Unit-integral normalization is the default so the figures compare distribution shapes.

Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

Display comparisons default to ROOT independent-error propagation. Shared-event and normalization correlations are not included. Binomial errors require an actual numerator-subset efficiency and are not appropriate for unfolded, smeared, normalized-shape, or F/B-to-F/B comparisons.

<!-- detailed-workflow-guide -->

### Detailed workflow and normalization

Reco, matched Ref, selected Ref, and inclusive Gen distributions diagnose how reconstruction and jet selections reshape the spectrum. Ratios such as Reco/Gen compare stored weighted yields unless a normalization mode is selected. Unit-area shape normalization uses $\tilde N_i=N_i/\sum_jN_j$; density normalization uses $\tilde N_i=N_i/(\Delta x_i\sum_jN_j)$.

The distinction matters for unequal bin widths. A shape ratio tests relative form after discarding total normalization, whereas a yield ratio retains rate differences. Lab, flipped-Lab, and CM eta coordinates are separate stored observables; the vertical acceptance lines are visual references and do not apply an additional selection.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [1]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from dataclasses import replace
from pathlib import Path
import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import SINGLE_JET_PT_BINS
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.plot_labels import CMS_SIMULATION_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, configure_cms_header, draw_text_block, save_canvas, set_2d_style,
    set_pad_style, set_palette_style,
)

CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = True  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


In [2]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

The defaults use the combined embedding file produced with the `jetId` file stem. `PT_BINS` controls the half-open projection intervals $[p_{T}^{min},p_{T}^{max})$. The histogram keys and style indices below are analysis inputs and are kept explicit for auditability.

In [3]:
SINGLE_JET_BINNING = 'normal'  # normal or test; independent of dijet binning
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
PT_BINS = tuple(pt_bins('single', SINGLE_JET_BINNING))
NORMALIZATION = 'integral'     # none, integral, or bin_width
NORMALIZATION_ETA_RANGE = (-1.0, 1.0) # (eta_low, eta_high), or None for full integral
RATIO_OPTION = 'B'  # preserved binomial setting; normalized shapes are not subset efficiencies
if RATIO_OPTION not in {'', 'B'}:
    raise ValueError("RATIO_OPTION must be '' or 'B'; B requires a true subset ratio")
RATIO_RANGE = (0.75, 1.15)
LOG_Y = True                  # logarithmic y axis for full distributions
ETA_RANGE = (-3.6, 3.6)              # e.g. (-3.0, 3.0), or None for the full axis
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'jet_selection'

FRAMES = ('Lab', 'CM')
HISTOGRAM_KEYS = {
    'Lab': {
        'Gen': 'hGenInclusiveJetPtEtaLab',
        'Reco': {
            'jetId': 'hRecoInclusiveJetPtEtaLab',
            'trkMax': 'hRecoInclusiveJetTrkMaxPtEtaLab',
            'noSel': 'hRecoInclusiveJetNoSelPtEtaLab',
        },
        'Ref': {
            'jetId': 'hRefInclusiveJetPtEtaLab',
            'trkMax': 'hRefInclusiveJetTrkMaxPtEtaLab',
            'noSel': 'hRefInclusiveJetNoSelPtEtaLab',
        },
        'RefSel': {
            'jetId': 'hRefSelInclusiveJetPtEtaLab',
            'trkMax': 'hRefSelInclusiveJetTrkMaxPtEtaLab',
            'noSel': 'hRefSelInclusiveJetNoSelPtEtaLab',
        },
    },
    'CM': {
        'Gen': 'hGenInclusiveJetPtEtaCM',
        'Reco': {
            'jetId': 'hRecoInclusiveJetPtEtaCM',
            'trkMax': 'hRecoInclusiveJetTrkMaxPtEtaCM',
            'noSel': 'hRecoInclusiveJetNoSelPtEtaCM',
        },
        'Ref': {
            'jetId': 'hRefInclusiveJetPtEtaCM',
            'trkMax': 'hRefInclusiveJetTrkMaxPtEtaCM',
            'noSel': 'hRefInclusiveJetNoSelPtEtaCM',
        },
        'RefSel': {
            'jetId': 'hRefSelInclusiveJetPtEtaCM',
            'trkMax': 'hRefSelInclusiveJetTrkMaxPtEtaCM',
            'noSel': 'hRefSelInclusiveJetNoSelPtEtaCM',
        },
    },
}
STYLE_INDICES = {'Reco': 0, 'Ref': 1, 'RefSel': 3, 'Gen': 2}
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE, annotation_text_size=0.025, legend_text_size=0.025,
)

def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE
SINGLE_JET_PT_BINS = pt_bins('single', SINGLE_JET_BINNING)

# Output identity uses the actual configured selections. Each execution gets
# dated renderer directories and unique filenames; earlier files are preserved.
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection, nominal_jet_eta_cut
OUTPUT_SELECTIONS = OutputSelection(tuple(dict.fromkeys(method for frame in FRAMES for keys in HISTOGRAM_KEYS[frame].values() if isinstance(keys, dict) for method in keys)), {'LabCMProjectionRange': ETA_RANGE or (), 'NormalizationRange': NORMALIZATION_ETA_RANGE or ()}, PT_BINS, 'jetPt')
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OUTPUT_SELECTIONS, directory_selection=FILE_STEM)



## Load and validate inputs

All objects must be two-dimensional histograms. Detached clones are retained after the input ROOT file is closed.

In [4]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
histograms_2d = {}
for frame in FRAMES:
    frame_keys = HISTOGRAM_KEYS[frame]
    histograms_2d[(frame, 'Gen')] = load_histogram(
        str(INPUT_FILE), frame_keys['Gen'],
    )
    for level in ('Reco', 'Ref', 'RefSel'):
        for selection, key in frame_keys[level].items():
            histograms_2d[(frame, f'{level} {selection}')] = load_histogram(
                str(INPUT_FILE), key,
            )
not_th2 = [label for label, histogram in histograms_2d.items()
           if not histogram.InheritsFrom('TH2')]
if not_th2:
    raise TypeError(f'Expected TH2 inputs for: {not_th2}')
{label: histogram.ClassName() for label, histogram in histograms_2d.items()}

{('Lab', 'Gen'): 'TH2D',
 ('Lab', 'Reco jetId'): 'TH2D',
 ('Lab', 'Reco trkMax'): 'TH2D',
 ('Lab', 'Reco noSel'): 'TH2D',
 ('Lab', 'Ref jetId'): 'TH2D',
 ('Lab', 'Ref trkMax'): 'TH2D',
 ('Lab', 'Ref noSel'): 'TH2D',
 ('Lab', 'RefSel jetId'): 'TH2D',
 ('Lab', 'RefSel trkMax'): 'TH2D',
 ('Lab', 'RefSel noSel'): 'TH2D',
 ('CM', 'Gen'): 'TH2D',
 ('CM', 'Reco jetId'): 'TH2D',
 ('CM', 'Reco trkMax'): 'TH2D',
 ('CM', 'Reco noSel'): 'TH2D',
 ('CM', 'Ref jetId'): 'TH2D',
 ('CM', 'Ref trkMax'): 'TH2D',
 ('CM', 'Ref noSel'): 'TH2D',
 ('CM', 'RefSel jetId'): 'TH2D',
 ('CM', 'RefSel trkMax'): 'TH2D',
 ('CM', 'RefSel noSel'): 'TH2D'}

## Two-dimensional inputs before projection

The Bird palette, logarithmic z scale, shared 2D axis styling, and palette placement follow `02_mc_event_histograms.ipynb`. Each map is shown before any $p_T$ range is applied.

In [5]:
# Cell role: draw the configured diagnostic figures and retain ROOT objects for display.
# Interpretation: Axis limits and logarithmic scales affect presentation only, not stored bin contents.
# The preceding Markdown gives the equations and physics assumptions for this step.
def draw_jet_map(histogram, frame, label, output_tag):
    plotted = histogram.Clone(f'{histogram.GetName()}_{output_tag}_2d')
    plotted.SetDirectory(0)
    plotted.SetTitle('')
    plotted.GetXaxis().SetTitle('p_{T}^{jet} (GeV)')
    eta_subscript = 'lab' if frame == 'Lab' else 'CM'
    plotted.GetYaxis().SetTitle(f'#eta_{{{eta_subscript}}}^{{jet}}')
    set_2d_style(plotted)

    canvas = ROOT.TCanvas(
        f'c_{output_tag}_2d', '',
        DEFAULT_PLOT_STYLE.canvas_width, DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
    plotted.Draw('COLZ')
    annotations = draw_text_block(canvas, (
        GENERATOR.capitalize(), DIRECTION, label, f'{frame} frame',
    ))
    canvas.Modified()
    canvas.Update()
    palette = set_palette_style(plotted)
    canvas.Modified()
    canvas.Update()
    save_canvas(
        canvas, OUTPUT_DIR / f'{output_tag}_2d.pdf', save_png=SAVE_PNG,
    )
    canvas._jet_map_objects = [plotted, palette, *annotations]
    return canvas

map_canvases = {}
for (frame, label), histogram in histograms_2d.items():
    output_tag = (
        f'{GENERATOR}_{DIRECTION}_{frame.lower()}_'
        f'{label.lower().replace(" ", "_")}'
    )
    canvas = draw_jet_map(histogram, frame, label, output_tag)
    map_canvases[(frame, label)] = canvas
    display(canvas)

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_lab_gen_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_lab_reco_jetid_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_lab_reco_trkmax_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_lab_reco_nosel_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_lab_ref_jetid_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_lab_ref_trkmax_2d_run1793907726755239000_8445d8e20e31.pdf has been created


... (10 outputs left out)

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_cm_ref_nosel_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_cm_refsel_jetid_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_cm_refsel_trkmax_2d_run1793907726755239000_8445d8e20e31.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPtBins471c78fff7a37d2d_embedding_combined_cm_refsel_nosel_2d_run1793907726755239000_8445d8e20e31.pdf has been created


## Lab- and CM-frame $\eta^{jet}$ projections and ratios to Gen

Each selection has one canvas per $p_T$ interval. Reco, matched Ref, RefSel, and inclusive Gen are overlaid in the upper panel; the three non-Gen curves share the ratio canvas. The default `NORMALIZATION='integral'` applies $1/h.Integral()$ independently to every projection. Set `NORMALIZATION_ETA_RANGE=(eta_low, eta_high)` to normalize instead with the half-open ROOT-bin integral from `FindBin(eta_low + 0.001)` through `FindBin(eta_high - 0.001)`. Leave it as `None` for the full in-range integral. Set `NORMALIZATION` to `'none'` for stored weighted yields or `'bin_width'` for unit-area densities; `NORMALIZATION_ETA_RANGE` is ignored for those modes. Reco/Gen, Ref/Gen, and RefSel/Gen display ratios default to independent errors; normalized shapes are not binomial efficiencies. Projection intervals are half-open.

In [6]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
def eta_projection(histogram, frame, label, selection, pt_range):
    low, high = pt_range
    return project_semantic_th2(
        histogram, 'eta', pt_range,
        name=f'h{label}{selection}Eta{frame}_{low:g}_{high:g}'.replace('.', 'p'),
    )

def add_cm_eta_boundaries(canvas, output):
    canvas.cd()
    histogram = next(obj for obj in canvas.GetListOfPrimitives()
                      if obj.InheritsFrom('TH1'))
    y_low, y_high = histogram.GetMinimum(), histogram.GetMaximum()
    lines = []
    for eta, style in ((2.4, 1), (1.9, 2)):
        for sign in (-1, 1):
            line = ROOT.TLine(sign * eta, y_low, sign * eta, y_high)
            line.SetLineColor(ROOT.kBlack)
            line.SetLineStyle(style)
            line.SetLineWidth(2)
            line.Draw('SAME')
            lines.append(line)
    legend = next((obj for obj in canvas.GetListOfPrimitives()
                   if obj.InheritsFrom('TLegend')), None)
    if legend is not None:
        legend.AddEntry(lines[0], '|#eta_{CM}| = 2.4', 'l')
        legend.AddEntry(lines[2], '|#eta_{CM}| = 1.9', 'l')
        legend.Draw()
    canvas._cm_eta_boundary_objects = [*lines, legend]
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)

comparison_results = {}
projection_specs = (
    (frame, pt_range, selection)
    for frame in FRAMES
    for pt_range in PT_BINS
    for selection in HISTOGRAM_KEYS[frame]['Reco']
)
for frame, pt_range, selection in projection_specs:
    eta_subscript = 'lab' if frame == 'Lab' else 'CM'
    eta_title = f'#eta_{{{eta_subscript}}}^{{jet}}'
    low, high = pt_range
    pt_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    raw_curves = {
        'Reco': eta_projection(
            histograms_2d[(frame, f'Reco {selection}')],
            frame, 'Reco', selection, pt_range,
        ),
        'Ref': eta_projection(
            histograms_2d[(frame, f'Ref {selection}')],
            frame, 'Ref', selection, pt_range,
        ),
        'RefSel': eta_projection(
            histograms_2d[(frame, f'RefSel {selection}')],
            frame, 'RefSel', selection, pt_range,
        ),
        'Gen': eta_projection(
            histograms_2d[(frame, 'Gen')],
            frame, 'Gen', selection, pt_range,
        ),
    }
    curves = {
        label: normalize_histogram(
            histogram, NORMALIZATION,
            integral_range=NORMALIZATION_ETA_RANGE,
        )
        for label, histogram in raw_curves.items()
    }
    annotations = (
        GENERATOR.capitalize(), DIRECTION, f'{frame} frame',
        f'{low:g} < p_{{T}}^{{jet}} < {high:g} GeV',
        selection,
    )
    output_tag = (
        f'{GENERATOR}_{DIRECTION}_{frame.lower()}_{selection}_'
        f'reco_ref_refsel_to_gen_pt_{pt_tag}'
    )
    y_title = {
        'none': f'dN/d{eta_title}',
        'integral': f'1/N dN/d{eta_title}',
        'bin_width': f'1/N dN/d{eta_title}',
    }[NORMALIZATION]
    canvas = draw_overlay(
        curves, title='',
        x_title=eta_title, y_title=y_title,
        log_y=LOG_Y,
        x_range=ETA_RANGE, annotations=annotations, grid=DRAW_GRID,
        legend_bounds=(0.45, 0.18, 0.70, 0.30),
        output=OUTPUT_DIR / f'{output_tag}.pdf', save_png=SAVE_PNG,
        style=PLOT_STYLE,
    )
    if frame == 'CM':
        add_cm_eta_boundaries(
            canvas, OUTPUT_DIR / f'{output_tag}.pdf',
        )
    # Draw all ratios together on one individual canvas, rather than in a bottom pad.
    ratios = {}
    for label, histogram in curves.items():
        if label == 'Gen':
            continue
        ratio = ratio_to_nominal(
            histogram, curves['Gen'], name=f'{output_tag}_{label}_ratio',
            option=RATIO_OPTION,
        )
        ratios[label] = ratio
    ratio_canvas = draw_overlay(
        {label: ratio for label, ratio in ratios.items()},
        title='', x_title=eta_title, y_title='Ratio to Gen',
        x_range=ETA_RANGE, y_range=RATIO_RANGE, reference_y=1.0,
        grid=DRAW_GRID, headroom=1.1, style_indices=STYLE_INDICES,
        annotations=annotations,
        legend_bounds=(0.45, 0.18, 0.70, 0.30),
        output=OUTPUT_DIR / f'{output_tag}_ratios.pdf', save_png=SAVE_PNG,
        canvas_name=f'{output_tag}_ratios',
        style=PLOT_STYLE,
    )
    if frame == 'CM':
        add_cm_eta_boundaries(
            ratio_canvas, OUTPUT_DIR / f'{output_tag}_ratios.pdf',
        )
    display(ratio_canvas)
    comparison_results[(frame, pt_range, selection)] = {
        'canvas': canvas, 'ratio_canvas': ratio_canvas, 'raw_histograms': raw_curves,
        'histograms': curves, 'ratios_to_gen': ratios,
    }
    display(canvas)

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt50to80_embedding_combined_lab_jetId_reco_ref_refsel_to_gen_pt_50_80_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt50to80_embedding_combined_lab_jetId_reco_ref_refsel_to_gen_pt_50_80_ratios_run1793907726755239000_8445d8e20e31.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt50to80_embedding_combined_lab_trkMax_reco_ref_refsel_to_gen_pt_50_80_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt50to80_embedding_combined_lab_trkMax_reco_ref_refsel_to_gen_pt_50_80_ratios_run1793907726755239000_8445d8e20e31.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt50to80_embedding_combined_lab_noSel_reco_ref_refsel_to_gen_pt_50_80_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt50to80_embedding_combined_lab_noSel_reco_ref_refsel_to_gen_pt_50_80_ratios_run1793907726755239000_8445d8e20e31.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_lab_jetId_reco_ref_refsel_to_gen_pt_180_200_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_lab_jetId_reco_ref_refsel_to_gen_pt_180_200_ratios_run1793907726755239000_8445d8e20e31.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_lab_trkMax_reco_ref_refsel_to_gen_pt_180_200_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_lab_trkMax_reco_ref_refsel_to_gen_pt_180_200_ratios_run1793907726755239000_8445d8e20e31.pdf has been created


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_lab_noSel_reco_ref_refsel_to_gen_pt_180_200_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_lab_noSel_reco_ref_refsel_to_gen_pt_180_200_ratios_run1793907726755239000_8445d8e20e31.pdf has been created


... (8 outputs left out)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_cm_noSel_reco_ref_refsel_to_gen_pt_180_200_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_cm_noSel_reco_ref_refsel_to_gen_pt_180_200_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt180to200_embedding_combined_cm_noSe

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_jetId_reco_ref_refsel_to_gen_pt_400_500_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_jetId_reco_ref_refsel_to_gen_pt_400_500_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_jetI

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_trkMax_reco_ref_refsel_to_gen_pt_400_500_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_trkMax_reco_ref_refsel_to_gen_pt_400_500_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_tr

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_overlay_0
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_noSel_reco_ref_refsel_to_gen_pt_400_500_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_noSel_reco_ref_refsel_to_gen_pt_400_500_run1793907726755239000_8445d8e20e31.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/jet_selection/2026-11-05/jetId/root/jetId_trkMax_noSel_etaLabCMProjectionRangem3p6_3p6_etaNormalizationRangem1_1_jetPt400to500_embedding_combined_cm_noSe

## Numerical summary

Print the raw in-range weighted yields before display normalization. These values provide a compact check of the input populations; they are not efficiencies unless the compared histograms share the intended denominator population.

In [7]:
# Cell role: perform analysis step 7.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
for (frame, pt_range, selection), result in comparison_results.items():
    yields = {
        label: histogram.Integral()
        for label, histogram in result['raw_histograms'].items()
    }
    print(
        f'{frame:3s} | {pt_range[0]:g}-{pt_range[1]:g} GeV | {selection:6s} '
        f'| Reco={yields["Reco"]:.6g} | Ref={yields["Ref"]:.6g} '
        f'| RefSel={yields["RefSel"]:.6g} '
        f'| Gen={yields["Gen"]:.6g}'
    )

Lab | 50-80 GeV | jetId  | Reco=0.0273143 | Ref=0.0272806 | RefSel=0.0244516 | Gen=0.0272321
Lab | 50-80 GeV | trkMax | Reco=0.0307011 | Ref=0.0306446 | RefSel=0.0271065 | Gen=0.0272321
Lab | 50-80 GeV | noSel  | Reco=0.0308044 | Ref=0.0307477 | RefSel=0.0272305 | Gen=0.0272321
Lab | 180-200 GeV | jetId  | Reco=4.43869e-05 | Ref=4.43863e-05 | RefSel=4.26517e-05 | Gen=4.37632e-05
Lab | 180-200 GeV | trkMax | Reco=4.58124e-05 | Ref=4.58117e-05 | RefSel=4.3441e-05 | Gen=4.37632e-05
Lab | 180-200 GeV | noSel  | Reco=4.6102e-05 | Ref=4.61014e-05 | RefSel=4.37627e-05 | Gen=4.37632e-05
Lab | 400-500 GeV | jetId  | Reco=1.21946e-06 | Ref=1.21946e-06 | RefSel=1.19051e-06 | Gen=1.19351e-06
Lab | 400-500 GeV | trkMax | Reco=1.21145e-06 | Ref=1.21144e-06 | RefSel=1.17821e-06 | Gen=1.19351e-06
Lab | 400-500 GeV | noSel  | Reco=1.22604e-06 | Ref=1.22603e-06 | RefSel=1.19351e-06 | Gen=1.19351e-06
CM  | 50-80 GeV | jetId  | Reco=0.0273143 | Ref=0.0272806 | RefSel=0.0244516 | Gen=0.027485
CM  | 50-80 G